# TFG V3.1 — Odd all-ones interference

This version keeps the V3 window-loader structure and tests a very small interference rule. Loaded all-one windows with odd indices receive a negative phase, then odd/even index pairs are mixed so equal window contents can interfere through the loader.

## Problem model

The search space is the valid range of candidate window-start indices. Each branch carries the candidate index and the corresponding loaded window:

```text
sum_i |grid>|idx=i>|window_i>
```

The grid is deterministic, so the probability report uses a reduced exact simulator over only `idx + m` while the full V3-style circuit is still built.

## Circuit structure

1. Prepare a uniform superposition over valid candidate indices.
2. Load `window_i` into the work register `m`.
3. Apply phase `-1` when `m=111...1` and the index is odd.
4. Mix odd/even pairs with `H(idx[0])`.
5. Apply the same XOR loader again so matching window contents collapse to `m=000...0`.
6. Compare the collapse flag with the physical free-window probability.

## Registers and data

- `n`: fixed grid with free and occupied cells.
- `idx`: candidate window-start index.
- `m`: loaded candidate window.
- `position_bits`: classical occupancy bitstring used by the reduced simulator.

## Purpose of this version

V3.1 isolates one specific interference idea: all-one odd-index branches are phase-flipped, mixed, and then collided with the loader. It is a diagnostic prototype, not a complete amplitude-amplification algorithm.

In [ ]:
import qiskit
import numpy as np

from math import prod
from qiskit import QuantumCircuit, QuantumRegister

print(qiskit.__version__)

In [ ]:
# =========================================================
# ND geometry and V3-style loading helpers
# =========================================================

def coord_to_index(coord, dims):
    idx_lin = 0
    stride = 1
    for d in reversed(range(len(dims))):
        idx_lin += coord[d] * stride
        stride *= dims[d]
    return idx_lin


def valid_starts_nd(N, M):
    shape_starts = tuple(N[d] - M[d] + 1 for d in range(len(N)))
    return list(np.ndindex(shape_starts))


def window_qubits_nd(start, N, M):
    qubits = []
    for offset in np.ndindex(tuple(M)):
        coord = tuple(start[d] + offset[d] for d in range(len(N)))
        qubits.append(coord_to_index(coord, N))
    return qubits


def reshape_bits_nd(bitstring, dims):
    return np.array(list(bitstring), dtype=str).reshape(dims)


def format_nd_array(arr, indent=0):
    if arr.ndim == 1:
        return "[" + " ".join(arr.tolist()) + "]"
    inner = [format_nd_array(np.array(sub), indent + 2) for sub in arr]
    sep = ",\n" + " " * indent
    return "[\n" + " " * indent + sep.join(inner) + "\n" + " " * max(indent - 2, 0) + "]"


def prepare_valid_index_superposition(qc, idx, W):
    amps = np.zeros(2 ** len(idx), dtype=complex)
    amps[:W] = 1 / np.sqrt(W)
    qc.initialize(amps, idx)


def apply_window_loader(qc, grid, idx, work, starts, N, M):
    """XOR-load window_i into work for each valid index i."""
    IDX = len(idx)
    gray_full = [t ^ (t >> 1) for t in range(2 ** IDX)]
    order_valid = [g for g in gray_full if g < len(starts)]
    current_zero_mask = [False] * IDX

    for i in order_valid:
        bits = [(i >> b) & 1 for b in range(IDX)]
        target_zero_mask = [bits[b] == 0 for b in range(IDX)]
        win = window_qubits_nd(starts[i], N, M)

        for b in range(IDX):
            if current_zero_mask[b] != target_zero_mask[b]:
                qc.x(idx[b])
                current_zero_mask[b] = target_zero_mask[b]

        for j, grid_pos in enumerate(win):
            qc.mcx(list(idx) + [grid[grid_pos]], work[j])

    for b in range(IDX):
        if current_zero_mask[b]:
            qc.x(idx[b])


def apply_phase_when_all_one(qc, qubits):
    """Apply a phase flip to |11...1> on the given qubits."""
    qubits = list(qubits)
    if len(qubits) == 1:
        qc.z(qubits[0])
        return
    target = qubits[-1]
    qc.h(target)
    qc.mcx(qubits[:-1], target)
    qc.h(target)


def apply_all_ones_odd_phase(qc, idx, work):
    """Apply phase -1 when the loaded window is all ones and the index is odd."""
    apply_phase_when_all_one(qc, list(work) + [idx[0]])

In [ ]:
# =========================================================
# Problem instance
# =========================================================

D = 2
N = [4, 4]
M = [2, 2]

N_tot = prod(N)
M_tot = prod(M)
starts = valid_starts_nd(N, M)
W = len(starts)
IDX = int(np.ceil(np.log2(W))) if W > 1 else 1

occupied_positions = [0, 2, 3, 4, 5, 8, 9, 14, 15]
position_bits = "".join("1" if i in occupied_positions else "0" for i in range(N_tot))

print(f"N={N}, M={M}, windows={W}, index qubits={IDX}")
print("Grid:")
print(format_nd_array(reshape_bits_nd(position_bits, N)))

print("\nCandidate windows:")
for i, start in enumerate(starts):
    bits = "".join(position_bits[q] for q in window_qubits_nd(start, N, M))
    label = "free" if bits == "0" * M_tot else "all-ones" if bits == "1" * M_tot else "mixed"
    print(f"{i:2d} start={start} bits={bits} ({label})")

In [ ]:
# =========================================================
# Reduced exact simulator over idx + m
# =========================================================

def physical_window_bits(index):
    if index >= W:
        return None
    return "".join(position_bits[q] for q in window_qubits_nd(starts[index], N, M))


def window_value(index):
    bits = physical_window_bits(index)
    if bits is None:
        return 0
    return sum(int(bit) << j for j, bit in enumerate(bits))


window_values = [window_value(i) for i in range(W)]
free_indices = [i for i in range(W) if physical_window_bits(i) == "0" * M_tot]
all_one_indices = [i for i in range(W) if physical_window_bits(i) == "1" * M_tot]


def initial_reduced_state():
    state = np.zeros(2 ** (IDX + M_tot), dtype=complex)
    for i in range(W):
        state[i] = 1 / np.sqrt(W)
    return state


def reduced_loader(state):
    out = np.zeros_like(state)
    for i in range(2 ** IDX):
        win = window_values[i] if i < W else 0
        for m_val in range(2 ** M_tot):
            amp = state[(m_val << IDX) | i]
            if abs(amp) > 0:
                out[((m_val ^ win) << IDX) | i] += amp
    return out


def reduced_all_ones_odd_phase(state):
    out = state.copy()
    all_ones = (1 << M_tot) - 1
    for i in range(1, 2 ** IDX, 2):
        out[(all_ones << IDX) | i] *= -1
    return out


def reduced_parity_mixer(state):
    out = np.zeros_like(state)
    inv_sqrt2 = 1 / np.sqrt(2)
    for m_val in range(2 ** M_tot):
        for i in range(0, 2 ** IDX, 2):
            a = state[(m_val << IDX) | i]
            b = state[(m_val << IDX) | (i | 1)]
            out[(m_val << IDX) | i] = (a + b) * inv_sqrt2
            out[(m_val << IDX) | (i | 1)] = (a - b) * inv_sqrt2
    return out


def metrics(state):
    values = {
        "P(index in valid range)": 0.0,
        "P(index outside range)": 0.0,
        "P(index is free window)": 0.0,
        "P(index is all-ones window)": 0.0,
        "P(m=0000 collapse flag)": 0.0,
        "P(m=1111)": 0.0,
    }
    all_ones = (1 << M_tot) - 1
    for i in range(2 ** IDX):
        for m_val in range(2 ** M_tot):
            p = float(abs(state[(m_val << IDX) | i]) ** 2)
            if p <= 1e-14:
                continue
            if i < W:
                values["P(index in valid range)"] += p
            else:
                values["P(index outside range)"] += p
            if i in free_indices:
                values["P(index is free window)"] += p
            if i in all_one_indices:
                values["P(index is all-ones window)"] += p
            if m_val == 0:
                values["P(m=0000 collapse flag)"] += p
            if m_val == all_ones:
                values["P(m=1111)"] += p
    return values


def print_metrics(title, state):
    print(f"\n{title}")
    print("-" * len(title))
    for key, value in metrics(state).items():
        print(f"{key:32s} {value:.6f}")


def print_index_distribution(title, state, tol=1e-12):
    print(f"\n{title}")
    print("-" * len(title))
    for i in range(2 ** IDX):
        p = sum(float(abs(state[(m_val << IDX) | i]) ** 2) for m_val in range(2 ** M_tot))
        if p <= tol:
            continue
        if i < W:
            bits = physical_window_bits(i)
            label = "free" if i in free_indices else "all-ones" if i in all_one_indices else "mixed"
            print(f"idx={i:2d} start={starts[i]} physical={bits} {label:8s} P={p:.6f}")
        else:
            print(f"idx={i:2d} INVALID                         P={p:.6f}")

In [ ]:
# =========================================================
# Build the V3-style circuit and score the first load
# =========================================================

grid = QuantumRegister(N_tot, "n")
idx = QuantumRegister(IDX, "i")
work = QuantumRegister(M_tot, "m")
qc = QuantumCircuit(grid, idx, work)

for pos in occupied_positions:
    qc.x(grid[pos])

prepare_valid_index_superposition(qc, idx, W)
apply_window_loader(qc, grid, idx, work, starts, N, M)
qc.barrier(label="loaded_windows")

state_loaded = reduced_loader(initial_reduced_state())
print_metrics("After first window load", state_loaded)
print_index_distribution("Index probabilities after first load", state_loaded)

qc.draw(output="text", fold=-1)

In [ ]:
# =========================================================
# V3.1 interference step
# =========================================================
# 1. Phase flip all-one windows only when the index is odd.
# 2. Mix odd/even neighbors with H on the parity bit idx[0].
# 3. Apply the loader again. Since the loader uses XOR, equal window-content
#    branches collapse to m=0000 and opposite-phase all-one branches can cancel.

apply_all_ones_odd_phase(qc, idx, work)
qc.barrier(label="phase_all_ones_odd")

qc.h(idx[0])
qc.barrier(label="parity_mixer")

apply_window_loader(qc, grid, idx, work, starts, N, M)
qc.barrier(label="second_loader_interference")

state_final = reduced_loader(reduced_parity_mixer(reduced_all_ones_odd_phase(state_loaded)))
print_metrics("After odd all-ones phase + parity mix + second loader", state_final)
print_index_distribution("Final index probabilities", state_final)

qc.draw(output="text", fold=-1)

## Notes

- The phase rule is deliberately narrow: it marks only all-one windows whose index is odd.
- The mixer is a single Hadamard on `idx[0]`, so it only mixes odd/even pairs.
- `P(m=0000)` is a collapse/interference flag after the second loader. It is not the same as `P(index is free window)`, which reports the physical free-window probability.